# Maximum Likelihood Estimation: The Inverse Probability Paradigm

## 1. Introduction
Maximum Likelihood Estimation (MLE) is the foundational optimization framework for parametric statistical inference. It operates on an inverse probability principle: rather than calculating the probability of data given fixed parameters, MLE identifies the parameters that maximize the probability of the fixed, observed data.

This notebook provides a complete guide, bridging theoretical math with practical Python implementations. We will explore the inverse mapping paradigm, log-likelihood transformations, optimization techniques, and connections to machine learning loss functions.

## 2. Setup and Imports
We begin by importing the necessary analytical and visualization libraries. Setting a random seed ensures that our synthetic data generation is reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize

# Configure pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set standard plotting style
plt.style.use('seaborn-v0_8-whitegrid')

# Set global random seed
np.random.seed(42)
print('Environment initialized successfully.')


## 3. Data Generation: The Bernoulli Process
To build intuition, let us evaluate competing hypotheses against empirical reality. Consider a sequence of independent Bernoulli trials (like coin flips).

We will generate a synthetic dataset representing 10 trials, with a true underlying success probability of 0.8.

In [ ]:
# Define true parameters
n_trials = 10
true_p = 0.8

# Generate binary observations
observed_data = np.random.binomial(n=1, p=true_p, size=n_trials)
k_successes = np.sum(observed_data)
n_failures = n_trials - k_successes

print(f'Raw Data Array: {observed_data}')
print(f'Total Trials (n): {n_trials}')
print(f'Total Successes (k): {k_successes}')
print(f'Total Failures: {n_failures}')


## 4. Evaluating Competing Hypotheses
If we hypothesize a fair process (p = 0.5), the probability of observing 8 successes is low. If we hypothesize a heavily biased process (p = 0.2), the observation is statistically negligible. 

Let us manually compute the likelihood for a few specific hypotheses. The likelihood function for a Bernoulli process is:
L(p | Data) = p^k * (1 - p)^(n - k)

In [ ]:
def compute_simple_likelihood(p, k, n):
    return (p ** k) * ((1.0 - p) ** (n - k))

hypotheses = [0.2, 0.5, 0.8, 0.9]
print('Evaluating the likelihood of different parameters:')
print('-' * 50)

for p_hyp in hypotheses:
    lik_value = compute_simple_likelihood(p_hyp, k_successes, n_trials)
    print(f'Hypothesis (p = {p_hyp:.1f}) -> Likelihood: {lik_value:.6f}')

print('\nConclusion: p = 0.8 provides the highest likelihood (it is the least surprising).')


## 5. The Log-Likelihood Transformation
In production systems, maximizing the raw likelihood L(p | Data) is computationally intractable. The product of many small probabilities leads to severe numerical underflow (the computer rounds the tiny decimal to exactly 0.0).

We apply the natural logarithm. Because log is a strictly increasing function, the parameter that maximizes the Likelihood also maximizes the Log-Likelihood.

ell(p | Data) = k * ln(p) + (n - k) * ln(1 - p)

In [ ]:
def compute_log_likelihood(p, k, n):
    # Clip to avoid log(0) domain errors
    p = np.clip(p, 1e-10, 1.0 - 1e-10)
    return k * np.log(p) + (n - k) * np.log(1.0 - p)

print('Evaluating the Log-Likelihood:')
print('-' * 50)

for p_hyp in hypotheses:
    ll_value = compute_log_likelihood(p_hyp, k_successes, n_trials)
    print(f'Hypothesis (p = {p_hyp:.1f}) -> Log-Likelihood: {ll_value:.4f}')


## 6. Visualizing the Parameter Space
Let us calculate the Likelihood and Log-Likelihood across the entire continuous parameter space (from 0 to 1) to visualize the objective functions we are trying to optimize.

In [ ]:
p_space = np.linspace(0.01, 0.99, 200)
L_vals = [compute_simple_likelihood(p, k_successes, n_trials) for p in p_space]
ll_vals = [compute_log_likelihood(p, k_successes, n_trials) for p in p_space]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(p_space, L_vals, color='blue', lw=2)
ax1.axvline(0.8, color='red', linestyle='--', label='Global Maximum')
ax1.set_title('Raw Likelihood Surface')
ax1.set_xlabel('Parameter p')
ax1.set_ylabel('Likelihood')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(p_space, ll_vals, color='green', lw=2)
ax2.axvline(0.8, color='red', linestyle='--', label='Global Maximum')
ax2.set_title('Log-Likelihood Surface')
ax2.set_xlabel('Parameter p')
ax2.set_ylabel('Log-Likelihood')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Notice how the maximum peak occurs at the exact same x-coordinate in both plots.')


## 7. The Score Function
The first derivative of the log-likelihood function with respect to the parameter is called the Score function. Finding the MLE analytically requires finding the roots of the Score function (where it crosses zero).

Score(p) = d/dp [ k*ln(p) + (n-k)*ln(1-p) ]
Score(p) = k/p - (n-k)/(1-p)

In [ ]:
score_vals = (k_successes / p_space) - ((n_trials - k_successes) / (1.0 - p_space))

plt.figure(figsize=(8, 5))
plt.plot(p_space, score_vals, color='purple', lw=2, label='Score Function')
plt.axhline(0, color='black', linestyle='-', lw=1)
plt.axvline(0.8, color='red', linestyle='--', label='Root (MLE = 0.8)')
plt.title('Score Function (First Derivative)')
plt.xlabel('Parameter p')
plt.ylabel('Gradient')
plt.ylim(-50, 50)
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 8. Asymptotic Consistency
A critical property of MLE is consistency: as the sample size (n) approaches infinity, the MLE estimate converges perfectly to the true underlying parameter.

Let us simulate this by drawing samples of increasing sizes and watching the variance of our estimates shrink.

In [ ]:
sample_sizes = [10, 50, 100, 500, 1000]
n_simulations = 100
simulation_results = []

for size in sample_sizes:
    for _ in range(n_simulations):
        data = np.random.binomial(n=1, p=true_p, size=size)
        mle_estimate = np.mean(data)
        simulation_results.append({'Sample Size': size, 'MLE Estimate': mle_estimate})

df_sim = pd.DataFrame(simulation_results)

plt.figure(figsize=(10, 6))
sns.boxplot(x='Sample Size', y='MLE Estimate', data=df_sim, color='lightblue')
plt.axhline(true_p, color='red', linestyle='--', label='True Parameter')
plt.title('Asymptotic Consistency of MLE for Bernoulli Distribution')
plt.xlabel('Sample Size (n)')
plt.ylabel('Estimated parameter')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()
print('The variance of the estimates shrinks toward zero around the true parameter as N grows.')


## 9. Machine Learning Connection: MSE is Gaussian MLE
MLE is the theoretical justification for Empirical Risk Minimization. If you assume the target variable y is generated by a deterministic function plus Gaussian noise, maximizing the likelihood of the data is mathematically identical to minimizing the Mean Squared Error (MSE).

Let us generate a linear regression dataset to prove this.

In [ ]:
n_points = 100
true_slope = 3.5
true_intercept = 10.0
true_sigma = 2.0

X_reg = np.random.uniform(0, 10, n_points)
noise = np.random.normal(0, true_sigma, n_points)
y_reg = true_slope * X_reg + true_intercept + noise

print('Synthetic linear regression data generated.')
print(f'True Parameters -> Slope: {true_slope}, Intercept: {true_intercept}')


## 10. Optimization Formulation
We will define a custom Negative Log-Likelihood (NLL) function for the Gaussian distribution. Since standard optimizers find minimums, we minimize the NLL, which is identical to maximizing the Log-Likelihood.

In [ ]:
def gaussian_nll(params, X, y):
    slope, intercept, sigma = params
    
    # Boundary constraint: Standard deviation must be positive
    if sigma <= 0:
        return np.inf
        
    # Predicted means given the current slope and intercept
    y_pred = slope * X + intercept
    
    # Sum of log-pdfs
    nll = -np.sum(stats.norm.logpdf(y, loc=y_pred, scale=sigma))
    return nll

initial_guess = [0.0, 0.0, 1.0]
param_bounds = [(None, None), (None, None), (1e-5, None)]

res_reg = minimize(gaussian_nll, initial_guess, args=(X_reg, y_reg), method='L-BFGS-B', bounds=param_bounds)
mle_slope, mle_intercept, mle_sigma = res_reg.x

print('Optimization via MLE completed.')
print(f'MLE Slope: {mle_slope:.4f}')
print(f'MLE Intercept: {mle_intercept:.4f}')
print(f'MLE Noise StdDev: {mle_sigma:.4f}')


## 11. Comparing MLE to Ordinary Least Squares (OLS)
To verify our claim, we will fit an OLS Linear Regression model from Scikit-Learn (which explicitly minimizes Mean Squared Error) and compare the resulting parameters.

In [ ]:
from sklearn.linear_model import LinearRegression

ols_model = LinearRegression()
ols_model.fit(X_reg.reshape(-1, 1), y_reg)
ols_slope = ols_model.coef_[0]
ols_intercept = ols_model.intercept_

print('--- Parameter Comparison ---')
print(f'MLE Slope: {mle_slope:.6f} | OLS Slope: {ols_slope:.6f}')
print(f'MLE Interc.: {mle_intercept:.6f} | OLS Interc.: {ols_intercept:.6f}')
print('\nThe parameters are identical, proving that minimizing MSE is maximizing Gaussian Likelihood.')


## 12. Visualizing the Regression
We overlay our MLE-derived regression line on top of the synthetic scatterplot.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_reg, y_reg, alpha=0.5, color='gray', label='Observed Data')

x_line = np.linspace(0, 10, 100)
plt.plot(x_line, true_slope * x_line + true_intercept, 'k--', lw=2, label='True Generative Line')
plt.plot(x_line, mle_slope * x_line + mle_intercept, 'b-', lw=2, label='MLE Fitted Line')

plt.title('Linear Regression via Maximum Likelihood')
plt.xlabel('X (Feature)')
plt.ylabel('y (Target)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 13. Edge Case: Boundary Parameters
Standard calculus assumes the maximum is in the interior. If the support of the distribution depends on the parameter (e.g., a Uniform distribution from 0 to Theta), the maximum occurs on the boundary.

For a Uniform(0, Theta) distribution, the derivative is never zero. The MLE is simply the maximum observed value in the dataset.

In [ ]:
true_theta_bound = 42.0
uniform_data = np.random.uniform(0, true_theta_bound, size=50)

# Analytical MLE for Uniform is the maximum element
mle_theta_bound = np.max(uniform_data)

print('Uniform Distribution Estimation:')
print(f'True Theta: {true_theta_bound}')
print(f'MLE Theta (Max Observed): {mle_theta_bound:.4f}')


## 14. Visualizing the Boundary Likelihood
Let us plot the likelihood function for the Uniform distribution to see why standard derivatives fail.

In [ ]:
theta_range = np.linspace(mle_theta_bound, 60, 200)
# Likelihood is (1/Theta)^n
uniform_likelihoods = (1.0 / theta_range) ** len(uniform_data)

plt.figure(figsize=(8, 5))
plt.plot(theta_range, uniform_likelihoods, color='red', lw=2)
plt.axvline(mle_theta_bound, color='black', linestyle='--', label='Boundary MLE')
plt.title('Likelihood Function for Uniform Distribution')
plt.xlabel('Parameter Theta')
plt.ylabel('Likelihood')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print('The maximum is at the extreme left edge. The derivative is strictly negative everywhere else.')


## 15. Practice Exercise: Poisson Distribution
The Poisson distribution models count data (e.g., website clicks per minute) using a single parameter, lambda.

Task: Use the synthetic dataset below to find the MLE for lambda numerically, and verify it against the analytical solution (which is known to be the sample mean).

In [ ]:
true_lambda_rate = 14.5
poisson_data = np.random.poisson(lam=true_lambda_rate, size=250)

def poisson_nll(params, data):
    lam = params[0]
    if lam <= 0:
        return np.inf
    return -np.sum(stats.poisson.logpmf(data, mu=lam))

res_poisson = minimize(poisson_nll, x0=[1.0], args=(poisson_data,), method='Nelder-Mead')
numerical_lambda = res_poisson.x[0]
analytical_lambda = np.mean(poisson_data)

print('Poisson Distribution Exercise Results:')
print(f'Numerical MLE via Scipy: {numerical_lambda:.4f}')
print(f'Analytical MLE (Mean):   {analytical_lambda:.4f}')
print('Both approaches yield the exact same parameter estimate.')


## 16. Computational Performance: Log-Sum-Exp Trick
When computing log-likelihoods for models involving sums of exponentials (like Softmax), direct computation causes catastrophic overflow. The Log-Sum-Exp trick is a numerical stabilization technique.

Formula: ln(Sum(exp(x_i))) = c + ln(Sum(exp(x_i - c))), where c is the maximum value.

In [ ]:
large_values = np.array([1000.0, 1010.0, 1020.0])

print('Attempting direct computation: np.log(np.sum(np.exp(large_values)))')
with np.errstate(over='ignore'):
    direct_result = np.log(np.sum(np.exp(large_values)))
    print(f'Direct Result: {direct_result} (Overflow to Infinity)\n')

c_max = np.max(large_values)
safe_result = c_max + np.log(np.sum(np.exp(large_values - c_max)))

print('Applying Log-Sum-Exp Stabilization:')
print(f'Safe Result: {safe_result:.4f}')


## 17. 3D Likelihood Surface Visualization
To truly understand optimization, we must visualize the multi-parameter landscape. We will plot the 3D Negative Log-Likelihood surface for the residuals of our linear regression.

In [ ]:
residuals = y_reg - (mle_slope * X_reg + mle_intercept)

mu_grid = np.linspace(-1.5, 1.5, 30)
sig_grid = np.linspace(0.5, 3.5, 30)
MU, SIGMA = np.meshgrid(mu_grid, sig_grid)
Z_NLL = np.zeros_like(MU)

for i in range(MU.shape[0]):
    for j in range(MU.shape[1]):
        Z_NLL[i, j] = gaussian_nll([0.0, MU[i, j], SIGMA[i, j]], np.zeros_like(residuals), residuals)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(MU, SIGMA, Z_NLL, cmap='viridis', alpha=0.8)

ax.set_title('3D Negative Log-Likelihood Valley')
ax.set_xlabel('Mean Parameter')
ax.set_ylabel('Sigma Parameter')
ax.set_zlabel('Negative Log-Likelihood')
plt.tight_layout()
plt.show()
print('The optimization algorithm descends into the lowest point of this valley.')


## 18. Summary and Key Takeaways
- **Inverse Principle**: MLE finds the parameters that maximize the probability of the fixed, observed data. It moves backward from data to model.
- **Log Transformation**: The Log-Likelihood transformation is mandatory to convert products into sums, preventing numerical underflow and simplifying derivatives.
- **Machine Learning**: Standard loss functions (MSE, Cross-Entropy) are simply Negative Log-Likelihoods derived from specific statistical assumptions.
- **Boundary Conditions**: Analytical derivatives fail when the optimal parameter lies on the boundary of the parameter space.